In [1]:
# ============================================================
# Deployment Test - Imports
# ============================================================

from pathlib import Path

import joblib
import json
import pandas as pd
import numpy as np


print("Deployment libraries imported successfully.")

Deployment libraries imported successfully.


In [2]:
# ============================================================
# Load Saved Experiment 2 Model
# ============================================================

MODEL_DIR = Path('../../../models/experiment_2')

MODEL_PATH = (
    MODEL_DIR /
    'gradient_boosting_top4_future_value.joblib'
)

METADATA_PATH = (
    MODEL_DIR /
    'gradient_boosting_top4_future_value_metadata.json'
)


# ------------------------------------------------------------
# Check files exist
# ------------------------------------------------------------

print("Model exists   :", MODEL_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())


# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

final_model = joblib.load(
    MODEL_PATH
)


# ------------------------------------------------------------
# Load metadata
# ------------------------------------------------------------

with open(
    METADATA_PATH,
    'r',
    encoding='utf-8'
) as file:

    model_metadata = json.load(file)


print("\nModel loaded successfully.")
print("\nModel type:")
print(type(final_model))

print("\nPipeline steps:")
print(final_model.named_steps.keys())

Model exists   : True
Metadata exists: True

Model loaded successfully.

Model type:
<class 'sklearn.pipeline.Pipeline'>

Pipeline steps:
dict_keys(['preprocessor', 'regressor'])


In [3]:
# ============================================================
# Verify Model Metadata
# ============================================================

print("=" * 70)
print("SAVED MODEL INFORMATION")
print("=" * 70)

print(
    "\nExperiment:",
    model_metadata['experiment']
)

print(
    "\nTask:",
    model_metadata['task']
)

print(
    "\nModel:",
    model_metadata['model']
)

print(
    "\nForecast horizon:",
    model_metadata['forecast_horizon']
)

print(
    "\nNumber of features:",
    model_metadata['n_features']
)

print("\nRequired features:")

for feature in model_metadata['features']:
    print(f" - {feature}")

print("\nSaved test metrics:")

print(
    f"R²   : "
    f"{model_metadata['test_metrics']['r2']:.4f}"
)

print(
    f"MAE  : "
    f"€{model_metadata['test_metrics']['mae_eur']:,.0f}"
)

print(
    f"RMSE : "
    f"€{model_metadata['test_metrics']['rmse_eur']:,.0f}"
)

SAVED MODEL INFORMATION

Experiment: Experiment 2 - Future Market Value Prediction

Task: Predict football player market value approximately six months ahead

Model: GradientBoostingRegressor

Forecast horizon: approximately 6 months

Number of features: 4

Required features:
 - current_market_value
 - age_at_snapshot
 - minutes_last_6m
 - goals_last_6m

Saved test metrics:
R²   : 0.9175
MAE  : €1,423,911
RMSE : €3,566,904


In [5]:
# ============================================================
# Check Raw Data Files
# ============================================================

RAW_DATA_DIR = Path('../../../data/raw')

print("Raw data folder exists:", RAW_DATA_DIR.exists())

print("\nFiles inside data/raw:")
print("-" * 70)

if RAW_DATA_DIR.exists():

    for file in sorted(RAW_DATA_DIR.iterdir()):
        print(file.name)

else:
    print("data/raw folder was not found.")

Raw data folder exists: True

Files inside data/raw:
----------------------------------------------------------------------
appearances.csv
club_games.csv
clubs.csv
competitions.csv
countries.csv
df_final.csv
game_events.csv
game_lineups.csv
games.csv
national_teams.csv
player_valuations.csv
players.csv
transfers.csv


In [6]:
# ============================================================
# Inspect Deployment Data Sources
# ============================================================

# Temporal dataset used by Experiment 2
TEMPORAL_DATA_PATH = Path(
    '../../../data/processed/temporal_market_value_dataset.csv'
)

# Original player information dataset
PLAYERS_DATA_PATH = Path(
    '../../../data/raw/players.csv'
)


print("Temporal dataset exists:", TEMPORAL_DATA_PATH.exists())
print("Players dataset exists :", PLAYERS_DATA_PATH.exists())


# ------------------------------------------------------------
# Load datasets
# ------------------------------------------------------------

temporal_df = pd.read_csv(
    TEMPORAL_DATA_PATH
)

players_df = pd.read_csv(
    PLAYERS_DATA_PATH
)


# ------------------------------------------------------------
# Basic information
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TEMPORAL DATASET")
print("=" * 70)

print("Shape:", temporal_df.shape)

print("\nColumns:")

for column in temporal_df.columns:
    print(f" - {column}")


print("\n" + "=" * 70)
print("PLAYERS DATASET")
print("=" * 70)

print("Shape:", players_df.shape)

print("\nColumns:")

for column in players_df.columns:
    print(f" - {column}")

Temporal dataset exists: True
Players dataset exists : True

TEMPORAL DATASET
Shape: (213905, 26)

Columns:
 - player_id
 - snapshot_date
 - target_date
 - future_date
 - target_distance_days
 - performance_window_start
 - current_market_value
 - age_at_snapshot
 - games_last_6m
 - goals_last_6m
 - assists_last_6m
 - minutes_last_6m
 - yellow_cards_last_6m
 - red_cards_last_6m
 - goals_per_90_last_6m
 - assists_per_90_last_6m
 - goal_contribution_per_90_last_6m
 - minutes_per_game_last_6m
 - height_in_cm
 - position
 - sub_position
 - foot
 - country_of_citizenship
 - future_market_value
 - future_value_change_eur
 - future_value_change_pct

PLAYERS DATASET
Shape: (47669, 26)

Columns:
 - player_id
 - first_name
 - last_name
 - name
 - last_season
 - current_club_id
 - player_code
 - country_of_birth
 - city_of_birth
 - country_of_citizenship
 - date_of_birth
 - sub_position
 - position
 - foot
 - height_in_cm
 - contract_expiration_date
 - agent_name
 - image_url
 - international_caps

In [7]:
# ============================================================
# Inspect Player Profile Fields
# ============================================================

print("=" * 70)
print("PLAYER PROFILE FIELDS")
print("=" * 70)

# Fields we would like to use in the dashboard
profile_fields = [
    'player_id',
    'name',
    'first_name',
    'last_name',
    'image_url',
    'position',
    'sub_position',
    'current_club_id',
    'current_club_name',
    'country_of_citizenship',
    'date_of_birth'
]

for field in profile_fields:

    if field in players_df.columns:
        print(f"{field:<30} : YES")

    else:
        print(f"{field:<30} : NO")


print("\n" + "=" * 70)
print("SAMPLE PLAYER RECORDS")
print("=" * 70)

available_profile_fields = [
    field
    for field in profile_fields
    if field in players_df.columns
]

display(
    players_df[
        available_profile_fields
    ].head(10)
)

PLAYER PROFILE FIELDS
player_id                      : YES
name                           : YES
first_name                     : YES
last_name                      : YES
image_url                      : YES
position                       : YES
sub_position                   : YES
current_club_id                : YES
current_club_name              : YES
country_of_citizenship         : YES
date_of_birth                  : YES

SAMPLE PLAYER RECORDS


,player_id,name,first_name,last_name,image_url,position,sub_position,current_club_id,current_club_name,country_of_citizenship,date_of_birth
0,10,Miroslav Klose,Miroslav,Klose,https://img.a.transfermarkt.technology/portrai...,Attack,Centre-Forward,398,Società Sportiva Lazio S.p.A.,Germany,1978-06-09 00:00:00
1,26,Roman Weidenfeller,Roman,Weidenfeller,https://img.a.transfermarkt.technology/portrai...,Goalkeeper,Goalkeeper,16,Borussia Dortmund,Germany,1980-08-06 00:00:00
2,65,Dimitar Berbatov,Dimitar,Berbatov,https://img.a.transfermarkt.technology/portrai...,Attack,Centre-Forward,1091,Panthessalonikios Athlitikos Omilos Konstantin...,Bulgaria,1981-01-30 00:00:00
3,77,Lúcio,NaN,Lúcio,https://img.a.transfermarkt.technology/portrai...,Defender,Centre-Back,506,Juventus Football Club,Brazil,1978-05-08 00:00:00
4,80,Tom Starke,Tom,Starke,https://img.a.transfermarkt.technology/portrai...,Goalkeeper,Goalkeeper,27,FC Bayern München,Germany,1981-03-18 00:00:00
5,109,Dedê,NaN,Dedê,https://img.a.transfermarkt.technology/portrai...,Defender,Left-Back,825,Eskisehirspor,Brazil,1978-04-18 00:00:00
6,123,Christoph Metzelder,Christoph,Metzelder,https://img.a.transfermarkt.technology/portrai...,Defender,Centre-Back,33,FC Schalke 04,Germany,1980-11-05 00:00:00
7,132,Tomas Rosicky,Tomas,Rosicky,https://img.a.transfermarkt.technology/portrai...,Midfield,Attacking Midfield,11,Arsenal Football Club,Czech Republic,1980-10-04 00:00:00
8,162,Marc Ziegler,Marc,Ziegler,https://img.a.transfermarkt.technology/portrai...,Goalkeeper,Goalkeeper,79,Verein für Bewegungsspiele Stuttgart 1893,Germany,1976-06-13 00:00:00
9,215,Roque Santa Cruz,Roque,Santa Cruz,https://img.a.transfermarkt.technology/portrai...,Attack,Centre-Forward,1084,Málaga CF,Paraguay,1981-08-16 00:00:00


In [8]:
# ============================================================
# Build Latest Player Prediction Table
# ============================================================

# ------------------------------------------------------------
# 1. Convert snapshot date to datetime
# ------------------------------------------------------------

temporal_df['snapshot_date'] = pd.to_datetime(
    temporal_df['snapshot_date']
)


# ------------------------------------------------------------
# 2. Sort snapshots chronologically
# ------------------------------------------------------------

temporal_sorted = temporal_df.sort_values(
    ['player_id', 'snapshot_date']
)


# ------------------------------------------------------------
# 3. Keep latest available snapshot for each player
# ------------------------------------------------------------

latest_snapshots = (
    temporal_sorted
    .groupby('player_id', as_index=False)
    .tail(1)
    .copy()
)


# ------------------------------------------------------------
# 4. Select player profile information
# ------------------------------------------------------------

player_profile_columns = [
    'player_id',
    'name',
    'image_url',
    'position',
    'sub_position',
    'current_club_name',
    'country_of_citizenship',
    'date_of_birth'
]

player_profiles = (
    players_df[player_profile_columns]
    .drop_duplicates(subset='player_id')
    .copy()
)


# ------------------------------------------------------------
# 5. Merge profile + latest temporal snapshot
# ------------------------------------------------------------

player_prediction_df = latest_snapshots.merge(
    player_profiles,
    on='player_id',
    how='left',
    suffixes=('', '_profile')
)


# ------------------------------------------------------------
# 6. Check required model features
# ------------------------------------------------------------

required_features = model_metadata['features']

valid_prediction_mask = (
    player_prediction_df[required_features]
    .notna()
    .all(axis=1)
)

player_prediction_df['prediction_ready'] = (
    valid_prediction_mask
)


# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("=" * 70)
print("LATEST PLAYER PREDICTION TABLE")
print("=" * 70)

print(
    f"\nUnique latest player snapshots : "
    f"{len(latest_snapshots):,}"
)

print(
    f"Players after profile merge    : "
    f"{len(player_prediction_df):,}"
)

print(
    f"Prediction-ready players       : "
    f"{player_prediction_df['prediction_ready'].sum():,}"
)

print(
    f"Missing player names           : "
    f"{player_prediction_df['name'].isna().sum():,}"
)

print(
    f"Missing image URLs             : "
    f"{player_prediction_df['image_url'].isna().sum():,}"
)

print(
    "\nLatest snapshot date:",
    player_prediction_df['snapshot_date'].max().date()
)

LATEST PLAYER PREDICTION TABLE

Unique latest player snapshots : 24,275
Players after profile merge    : 24,275
Prediction-ready players       : 24,254
Missing player names           : 0
Missing image URLs             : 0

Latest snapshot date: 2025-12-22


In [9]:
# ============================================================
# Search and Predict One Player
# ============================================================

# ------------------------------------------------------------
# 1. Choose a player to test
# ------------------------------------------------------------

SEARCH_NAME = "Cole Palmer"


# ------------------------------------------------------------
# 2. Search player
# ------------------------------------------------------------

search_results = player_prediction_df[
    player_prediction_df['name']
    .str.contains(
        SEARCH_NAME,
        case=False,
        na=False
    )
].copy()


print("=" * 70)
print("PLAYER SEARCH RESULTS")
print("=" * 70)

print(f"\nSearch: {SEARCH_NAME}")
print(f"Players found: {len(search_results)}")


display_columns = [
    'player_id',
    'name',
    'position',
    'sub_position',
    'current_club_name',
    'snapshot_date',
    'prediction_ready'
]

display(
    search_results[
        display_columns
    ]
)

PLAYER SEARCH RESULTS

Search: Cole Palmer
Players found: 1


,player_id,name,position,sub_position,current_club_name,snapshot_date,prediction_ready
19904,568177,Cole Palmer,Midfield,Attacking Midfield,Chelsea Football Club,2025-05-30,True


In [10]:
# ============================================================
# Predict Selected Player
# ============================================================

if len(search_results) == 0:

    print("No player found.")

else:

    # --------------------------------------------------------
    # 1. Select first matching player
    # --------------------------------------------------------

    selected_player = (
        search_results
        .sort_values('snapshot_date')
        .iloc[-1]
    )


    # --------------------------------------------------------
    # 2. Prepare the four model features
    # --------------------------------------------------------

    player_features = pd.DataFrame([{
        'current_market_value':
            selected_player['current_market_value'],

        'age_at_snapshot':
            selected_player['age_at_snapshot'],

        'minutes_last_6m':
            selected_player['minutes_last_6m'],

        'goals_last_6m':
            selected_player['goals_last_6m']
    }])


    # --------------------------------------------------------
    # 3. Predict future market value
    # --------------------------------------------------------

    predicted_future_value = final_model.predict(
        player_features
    )[0]


    # --------------------------------------------------------
    # 4. Calculate predicted change
    # --------------------------------------------------------

    current_value = selected_player[
        'current_market_value'
    ]

    predicted_change = (
        predicted_future_value
        - current_value
    )

    predicted_change_pct = (
        predicted_change
        / current_value
        * 100
    )


    # --------------------------------------------------------
    # 5. Display result
    # --------------------------------------------------------

    print("=" * 70)
    print("PLAYER MARKET VALUE FORECAST")
    print("=" * 70)

    print(
        f"\nPlayer       : "
        f"{selected_player['name']}"
    )

    print(
        f"Player ID    : "
        f"{selected_player['player_id']}"
    )

    print(
        f"Position     : "
        f"{selected_player['position']}"
    )

    print(
        f"Sub-position : "
        f"{selected_player['sub_position']}"
    )

    print(
        f"Club         : "
        f"{selected_player['current_club_name']}"
    )

    print(
        f"Snapshot Date: "
        f"{selected_player['snapshot_date'].date()}"
    )

    print("\nMODEL INPUTS")
    print("-" * 70)

    print(
        f"Current Market Value : "
        f"€{current_value:,.0f}"
    )

    print(
        f"Age                  : "
        f"{selected_player['age_at_snapshot']:.1f}"
    )

    print(
        f"Minutes Last 6 Months: "
        f"{selected_player['minutes_last_6m']:,.0f}"
    )

    print(
        f"Goals Last 6 Months  : "
        f"{selected_player['goals_last_6m']:,.0f}"
    )

    print("\nFORECAST")
    print("-" * 70)

    print(
        f"Predicted Value (~6m): "
        f"€{predicted_future_value:,.0f}"
    )

    print(
        f"Predicted Change     : "
        f"€{predicted_change:+,.0f}"
    )

    print(
        f"Predicted Change (%) : "
        f"{predicted_change_pct:+.2f}%"
    )

    print(
        f"\nImage URL:"
        f"\n{selected_player['image_url']}"
    )

PLAYER MARKET VALUE FORECAST

Player       : Cole Palmer
Player ID    : 568177
Position     : Midfield
Sub-position : Attacking Midfield
Club         : Chelsea Football Club
Snapshot Date: 2025-05-30

MODEL INPUTS
----------------------------------------------------------------------
Current Market Value : €120,000,000
Age                  : 23.1
Minutes Last 6 Months: 2,243
Goals Last 6 Months  : 8

FORECAST
----------------------------------------------------------------------
Predicted Value (~6m): €124,546,897
Predicted Change     : €+4,546,897
Predicted Change (%) : +3.79%

Image URL:
https://img.a.transfermarkt.technology/portrait/header/568177-1712320986.jpg?lm=1


In [11]:
# ============================================================
# Check Training / Notebook Environment Versions
# ============================================================

import sys
import sklearn
import joblib

print("Python:")
print(sys.executable)

print("\nScikit-learn:")
print(sklearn.__version__)

print("\nJoblib:")
print(joblib.__version__)

Python:
c:\Users\ngtht\anaconda3\python.exe

Scikit-learn:
1.6.1

Joblib:
1.4.2


In [12]:
import sys
import sklearn
import joblib

print("Python:", sys.executable)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)

Python: c:\Users\ngtht\anaconda3\python.exe
Scikit-learn: 1.6.1
Joblib: 1.4.2


In [13]:
# ============================================================
# XAI Step 1 - Check SHAP Environment
# ============================================================

import sys

print("Python:", sys.executable)

try:
    import shap

    print("SHAP installed: Yes")
    print("SHAP version:", shap.__version__)

except ImportError:
    print("SHAP installed: No")

Python: c:\Users\ngtht\anaconda3\python.exe
SHAP installed: No


In [14]:
# ============================================================
# XAI Step 2 - Install SHAP
# ============================================================

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "shap"
])

0